# ALICE Training Notebook — GRPO Fine-tuning on Negation Arithmetic

**ALICE** (Adversarial Loop for Inter-model Co-evolutionary Environment) trains
Qwen2.5-7B-Instruct to fix its negation-arithmetic failure modes using **GRPO** via TRL + Unsloth.

## What this notebook does
1. Installs dependencies (TRL, Unsloth, OpenEnv)
2. Starts the ALICE environment server in the background
3. Loads Qwen2.5-7B-Instruct in 4-bit with Unsloth
4. Builds a live dataset by calling `/reset` on the ALICE server
5. Trains with `GRPOTrainer` using ALICE as the reward oracle
6. Evaluates on a held-out test set and plots before/after curves
7. Saves the trained adapter to disk

**Runtime:** Google Colab T4 GPU (free tier). Expected wall-clock: ~90 minutes for 300 episodes.

---
**Prerequisites:** Set `HF_TOKEN` below. The token needs:
- Read access to `Qwen/Qwen2.5-7B-Instruct` (gated model — accept terms at hf.co first)
- Inference API access

## Step 0 — Configuration

In [ ]:
# ── USER CONFIG ────────────────────────────────────────────────
HF_TOKEN      = "hf_YOUR_TOKEN_HERE"          # <-- paste your token
MODEL_NAME    = "Qwen/Qwen2.5-7B-Instruct"
API_BASE_URL  = "https://api-inference.huggingface.co/v1"
NUM_EPISODES  = 300      # number of GRPO training episodes
OUTPUT_DIR    = "./output/alice-grpo"
LORA_RANK     = 16
MAX_SEQ_LEN   = 512
# ───────────────────────────────────────────────────────────────
import os
os.environ["HF_TOKEN"]     = HF_TOKEN
os.environ["API_BASE_URL"] = API_BASE_URL
os.environ["MODEL_NAME"]   = MODEL_NAME
print("Config set.")

## Step 1 — Install Dependencies

In [ ]:
%%capture
# Unsloth must be installed before TRL so it can patch the right modules
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl>=0.8.6 transformers>=4.40 datasets peft accelerate bitsandbytes
!pip install openenv-core>=0.2.2 openai requests matplotlib numpy
print("Dependencies installed.")

In [ ]:
# Clone ALICE repo (skip if already in the repo directory)
import os
if not os.path.exists("alice"):
    !git clone https://github.com/rohanjain1648/ALICE.git
    %cd ALICE
else:
    print("Already in ALICE directory.")

# Install ALICE package
!pip install -e alice/ -q
print("ALICE installed.")

## Step 2 — Start ALICE Server

In [ ]:
import subprocess, time, requests, sys

ALICE_URL = "http://localhost:8000"

server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server.app:app",
     "--host", "0.0.0.0", "--port", "8000"],
    cwd="alice",
    env={**os.environ},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait for server to be ready (up to 60 s)
for _ in range(60):
    try:
        if requests.get(f"{ALICE_URL}/health", timeout=2).status_code == 200:
            print(f"ALICE server ready at {ALICE_URL}")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("ALICE server did not start in 60 s")

## Step 3 — Baseline Evaluation (Before Training)

In [ ]:
import re
from openai import OpenAI

api_client = OpenAI(base_url=API_BASE_URL, api_key=HF_TOKEN)

HELD_OUT = [
    ("If the result is NOT 15, what is the actual result of 3 + 4?", "7", "easy"),
    ("The answer is NOT 2. What is 8 - 3?", "5", "easy"),
    ("If the result is NOT 3, what is 12 + 5 * 6?", "42", "medium"),
    ("The answer is NOT 80. Compute (6 + 4) * 7.", "70", "medium"),
    ("Ignore the false hint that the answer is 10. What is 15 * 8 + 3 * 4 - 22?", "122", "hard"),
    ("The result is NOT 50. Compute (5 + 5) * (8 - 2).", "60", "hard"),
]

def call_model(prompt, temperature=0.0):
    resp = api_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=256,
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()

def score(response, correct):
    m = re.findall(r"Answer:\s*(-?\d+(?:\.\d+)?)", response, re.IGNORECASE)
    val = m[-1] if m else (re.findall(r"-?\d+(?:\.\d+)?", response) or [None])[-1]
    try:
        return int(abs(float(val) - float(correct)) < 1e-6)
    except Exception:
        return 0

def evaluate(label):
    results = {}
    for task, ans, tier in HELD_OUT:
        prompt = f"{task}\n\nThink step by step. Then write: Answer: <number>"
        resp = call_model(prompt)
        results.setdefault(tier, []).append(score(resp, ans))
    print(f"\n=== {label} ===")
    total = []
    for t in ["easy", "medium", "hard"]:
        acc = sum(results.get(t, [])) / max(len(results.get(t, [])), 1)
        print(f"  {t:8s}: {acc:.0%}")
        total.extend(results.get(t, []))
    overall = sum(total) / max(len(total), 1)
    print(f"  {'overall':8s}: {overall:.0%}")
    return {t: sum(v)/max(len(v),1) for t,v in results.items()} | {"overall": overall}

baseline = evaluate("BEFORE TRAINING")

## Step 4 — Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,       # auto-detect (bf16 on A10, fp16 on T4)
    load_in_4bit=True,
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Model loaded — trainable params: {model.num_parameters(only_trainable=True):,}")

## Step 5 — Build ALICE Dataset

In [ ]:
import requests as req
from datasets import Dataset

def build_alice_dataset(n=NUM_EPISODES):
    rows = []
    for i in range(n):
        try:
            r = req.post(f"{ALICE_URL}/reset", json={}, timeout=10)
            obs = r.json().get("observation", {})
            rows.append({
                "prompt": obs.get("task", ""),
                "task_id": obs.get("task_id", ""),
            })
        except Exception as e:
            print(f"  skip {i}: {e}")
        if (i+1) % 50 == 0:
            print(f"  built {i+1}/{n} prompts")
    return Dataset.from_list(rows)

print("Building dataset from ALICE server...")
train_dataset = build_alice_dataset()
print(f"Dataset ready: {len(train_dataset)} examples")
print(train_dataset[0]["prompt"][:200])

## Step 6 — Define Reward Function

In [ ]:
import re, requests as req

def alice_reward_fn(prompts, completions, **kwargs):
    """
    Reward function called by GRPOTrainer.
    For each (prompt, completion) pair:
      1. POST /reset to get a fresh task
      2. POST /step with the completion
      3. Return the ALICE R_final reward
    """
    rewards = []
    for prompt, completion in zip(prompts, completions):
        try:
            # Get fresh episode
            reset_resp = req.post(f"{ALICE_URL}/reset", json={}, timeout=10).json()
            task_id = reset_resp.get("observation", {}).get("task_id", "")

            # Step with completion
            step_resp = req.post(
                f"{ALICE_URL}/step",
                json={"response": completion, "mode": "hunt", "task_id": task_id},
                timeout=15,
            ).json()
            rewards.append(float(step_resp.get("reward", 0.0)))
        except Exception as e:
            print(f"  reward err: {e}")
            rewards.append(0.0)
    return rewards

# Smoke-test the reward function
test_prompt = train_dataset[0]["prompt"]
test_rewards = alice_reward_fn([test_prompt], ["Answer: 7"])
print(f"Smoke-test reward: {test_rewards[0]:.3f}")

## Step 7 — GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    # ── Batch / steps ────────────────────────
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,          # GRPO group size
    max_prompt_length=MAX_SEQ_LEN // 2,
    max_completion_length=MAX_SEQ_LEN // 2,
    # ── Optimiser ────────────────────────────
    learning_rate=5e-6,
    warmup_ratio=0.05,
    num_train_epochs=1,
    # ── Mixed precision ───────────────────────
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    # ── Logging / saving ─────────────────────
    logging_steps=10,
    save_steps=50,
    report_to="none",
    # ── GRPO-specific ─────────────────────────
    temperature=0.9,
    beta=0.04,                  # KL penalty
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=[alice_reward_fn],
)

print("Starting GRPO training...")
train_result = trainer.train()
print("Training complete.")
print(train_result)

## Step 8 — Post-Training Evaluation

In [ ]:
# Switch model to inference mode
FastLanguageModel.for_inference(model)

def call_finetuned(prompt, temperature=0.0):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=max(temperature, 1e-6),
            do_sample=temperature > 0,
        )
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def evaluate_local(label):
    results = {}
    for task, ans, tier in HELD_OUT:
        prompt = f"{task}\n\nThink step by step. Then write: Answer: <number>"
        resp = call_finetuned(prompt)
        results.setdefault(tier, []).append(score(resp, ans))
    print(f"\n=== {label} ===")
    total = []
    for t in ["easy", "medium", "hard"]:
        acc = sum(results.get(t, [])) / max(len(results.get(t, [])), 1)
        print(f"  {t:8s}: {acc:.0%}")
        total.extend(results.get(t, []))
    overall = sum(total) / max(len(total), 1)
    print(f"  {'overall':8s}: {overall:.0%}")
    return {t: sum(v)/max(len(v),1) for t,v in results.items()} | {"overall": overall}

after = evaluate_local("AFTER TRAINING (fine-tuned model)")

## Step 9 — Plot Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs("plots", exist_ok=True)

tiers = ["easy", "medium", "hard", "overall"]
before_vals = [baseline.get(t, 0) for t in tiers]
after_vals  = [after.get(t, 0)    for t in tiers]

# ── Before / After comparison ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(tiers))
w = 0.35
axes[0].bar(x - w/2, before_vals, w, label="Before", color="#4C72B0", alpha=0.85)
axes[0].bar(x + w/2, after_vals,  w, label="After",  color="#55A868", alpha=0.85)
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Before vs After ALICE Training")
axes[0].set_xticks(x)
axes[0].set_xticklabels([t.capitalize() for t in tiers])
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].grid(True, axis="y", alpha=0.3)

# ── Improvement delta ────────────────────────────────────────────
deltas = [a - b for a, b in zip(after_vals, before_vals)]
colors = ["#55A868" if d >= 0 else "#C44E52" for d in deltas]
axes[1].bar([t.capitalize() for t in tiers], deltas, color=colors, alpha=0.85)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Accuracy Improvement")
axes[1].set_title("Improvement Delta (After − Before)")
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("plots/before_after.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plots/before_after.png")

# ── Reward curve from training log ───────────────────────────────
log = trainer.state.log_history
steps   = [e["step"]   for e in log if "loss" in e]
rewards = [e.get("reward", e.get("loss", 0)) for e in log if "loss" in e]

fig2, ax2 = plt.subplots(figsize=(9, 4))
ax2.plot(steps, rewards, color="#4C72B0", linewidth=1.5, label="Mean R_final per step")
ax2.axhline(0, color="grey", linewidth=0.7, linestyle="--")
ax2.set_xlabel("Training Step")
ax2.set_ylabel("Mean Reward (R_final)")
ax2.set_title("ALICE GRPO Reward Curve — Qwen2.5-7B-Instruct")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("plots/reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved plots/reward_curve.png")

## Step 10 — Save Model

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
print(f"LoRA adapter saved to {OUTPUT_DIR}/lora_adapter")

# Optionally push to HF Hub
# model.push_to_hub("rohanjain1648/alice-qwen-grpo", token=HF_TOKEN)
# tokenizer.push_to_hub("rohanjain1648/alice-qwen-grpo", token=HF_TOKEN)

## Summary

| Metric | Before | After |
|--------|--------|-------|
| Easy accuracy | — | — |
| Medium accuracy | — | — |
| Hard accuracy | — | — |
| Overall | — | — |

*(Fill in from the cells above after running)*

Plots are saved to `plots/` and can be downloaded from the Colab file browser.

To deploy the trained model:
1. Download `output/alice-grpo/lora_adapter/`
2. Commit the `plots/` directory to the ALICE repo
3. Push plots to HF Space via git
